# Document Question-Answering System (RAG)

**Goal:** Build a Retrieval-Augmented Generation system that answers questions grounded in a
custom PDF, instead of relying purely on a language model's parametric knowledge.

**Pipeline:** PDF loading → chunking → embedding → vector search (FAISS) → generation (FLAN-T5).

**What this notebook adds beyond a minimal implementation:**
- Cosine-similarity retrieval (not raw L2), which is the standard choice for sentence embeddings.
- Retrieval scores surfaced alongside answers, so results are auditable rather than a black box.
- An out-of-scope test question, to check the system doesn't hallucinate when the answer isn't
  actually in the document.
- A simple automated grounding check (lexical overlap between answer and retrieved context) as a
  proxy for "is this answer actually supported by the source text."


## 1. Install Libraries

In [ ]:
!pip install -q langchain-community langchain-text-splitters \
    pypdf sentence-transformers faiss-cpu transformers


## 2. Import Libraries

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np

print("All imports successful!")


## 3. Upload and Load the PDF

Wrapped in a try/except so a missing or corrupt upload fails with a clear message instead of a
cryptic traceback three cells later.

In [ ]:
import os

try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = next(iter(uploaded))  # take whatever file was just uploaded
except ImportError:
    # Not running in Colab -- fall back to a local file in the working directory.
    candidates = [f for f in os.listdir() if f.lower().endswith(".pdf")]
    if not candidates:
        raise FileNotFoundError(
            "No PDF found. Place a .pdf file in the working directory, or run this in Colab."
        )
    pdf_path = candidates[0]

print(f"Using PDF: {pdf_path}")

try:
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
except Exception as e:
    raise RuntimeError(f"Failed to load PDF '{pdf_path}': {e}")

print(f"Loaded {len(documents)} page(s).")
print(documents[0].page_content[:500])


## 4. Split the Text into Chunks

**Chunk size / overlap rationale:** 500 characters is roughly 2-3 sentences for dense text --
small enough that FLAN-T5-base's limited context isn't overwhelmed with irrelevant material,
large enough to preserve a complete thought. 50-character overlap (10%) prevents a sentence from
being split exactly at a chunk boundary and losing meaning on both sides.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)
texts = [chunk.page_content for chunk in chunks]

lengths = [len(t) for t in texts]
print(f"Number of chunks: {len(texts)}")
print(f"Avg chunk length: {np.mean(lengths):.0f} chars  (min {min(lengths)}, max {max(lengths)})")
print("\n--- First chunk preview ---")
print(texts[0])


## 5. Create Embeddings

Using `all-MiniLM-L6-v2`: a small, fast sentence-transformer that's a common, well-benchmarked
default for semantic search tasks like this one.

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(texts, convert_to_numpy=True)

print("Embedding matrix shape:", embeddings.shape)


## 6. Build the FAISS Vector Index

**Why cosine similarity instead of raw L2 distance:** sentence-transformer embeddings are
typically compared by cosine similarity, since what matters is the *direction* of the embedding
(semantic meaning), not its magnitude. To get cosine similarity out of FAISS, we L2-normalize
every vector and then use an inner-product index (`IndexFlatIP`) -- inner product of unit vectors
equals cosine similarity. This is a small but meaningful correctness fix over using
`IndexFlatL2` directly on unnormalized embeddings.

In [ ]:
def normalize(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1e-10  # avoid divide-by-zero on a degenerate embedding
    return vectors / norms

normalized_embeddings = normalize(embeddings).astype("float32")

dimension = normalized_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)   # inner product == cosine similarity on unit vectors
index.add(normalized_embeddings)

print(f"FAISS index built: {index.ntotal} vectors of dimension {dimension}.")


## 7. Load the Language Model

**Why `text2text-generation`, not `text-generation`:** FLAN-T5 is an encoder-decoder
(seq2seq) model. `text-generation` is the pipeline task for decoder-only causal LMs (e.g. GPT-2).
Using the wrong task for FLAN-T5 doesn't error -- it silently produces empty or garbage output,
which is easy to miss and was the bug in the original version of this notebook.

In [ ]:
try:
    generator = pipeline("text2text-generation", model="google/flan-t5-base")
except Exception as e:
    raise RuntimeError(f"Failed to load language model: {e}")

print("Language model loaded.")


## 8. Reusable RAG Query Function

Retrieves the top-k chunks, builds a grounded prompt, generates an answer, and returns enough
detail (retrieved chunks + similarity scores) to audit *why* the model answered the way it did --
rather than just a bare string.

In [ ]:
def answer_question(query: str, k: int = 3, max_new_tokens: int = 150):
    """
    Retrieve the top-k most relevant chunks for `query` and use them as grounding
    context for the language model's answer.

    Generation uses beam search (num_beams=4) with a no-repeat n-gram constraint
    instead of greedy decoding. FLAN-T5-base with plain greedy decoding tends to
    truncate early or repeat phrases on longer contexts; beam search with
    no_repeat_ngram_size trades a bit of speed for materially more coherent answers.

    Returns a dict with the answer, retrieved chunks, and their similarity scores.
    """
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    query_embedding = normalize(query_embedding).astype("float32")

    scores, indices = index.search(query_embedding, k)
    retrieved_chunks = [texts[i] for i in indices[0]]
    retrieved_scores = [float(s) for s in scores[0]]

    context = "\n\n".join(retrieved_chunks)
    prompt = f"""Answer the question using only the context below. If the answer is not
contained in the context, say "I don't have enough information to answer that."

Context:
{context}

Question:
{query}

Answer:"""

    result = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )
    answer = result[0]["generated_text"].strip()

    return {
        "question": query,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks,
        "scores": retrieved_scores,
    }


## 9. Test Questions

Includes three questions answerable from the document, plus one **deliberately out-of-scope**
question. A well-grounded RAG system should decline to answer the last one rather than
hallucinate -- this is a much more informative test than only using easy, in-scope questions.

In [ ]:
queries = [
    "What are the objectives of this project?",
    "What is Retrieval-Augmented Generation (RAG)?",
    "What components are used to build the pipeline?",
    "What is the capital of France?",  # out-of-scope control question
]


## 10. Run the Pipeline on Each Question

In [ ]:
results = [answer_question(q) for q in queries]

for r in results:
    print("Question:", r["question"])
    print("Answer  :", r["answer"])
    print("Top match score:", round(r["scores"][0], 3))
    print("-" * 70)


## 11. Automated Grounding Check

**Why not lexical overlap alone:** the original version of this check counted shared words
between the answer and the retrieved context. That's cheap, but it fails in both directions --
it penalizes correct answers that simply paraphrase the source (no shared vocabulary), and it
would score a fluent hallucination highly if it happens to reuse context words without actually
being supported by them.

**Fix:** compute cosine similarity between the *embedding* of the answer and the embedding of
the retrieved context, using the same `all-MiniLM-L6-v2` model already loaded for retrieval. This
captures semantic support even when the wording differs, and is far harder to fool with
coincidental word reuse. Lexical overlap is kept alongside it as a secondary, more literal signal
-- the two together are more informative than either alone, and a low score on *both* is a much
stronger red flag than a low score on one.

In [ ]:
import re

def word_overlap_ratio(answer: str, context: str) -> float:
    """Fraction of the answer's content words that also appear in the retrieved context."""
    stopwords = {"the","a","an","is","are","of","to","in","and","that","this","it","i",
                 "have","don't","enough","information","answer"}
    ans_words = {w for w in re.findall(r"[a-z']+", answer.lower()) if w not in stopwords}
    ctx_words = set(re.findall(r"[a-z']+", context.lower()))
    if not ans_words:
        return 0.0
    return len(ans_words & ctx_words) / len(ans_words)


def embedding_grounding_score(answer: str, context: str) -> float:
    """Cosine similarity between the answer's embedding and the retrieved context's embedding."""
    vecs = embed_model.encode([answer, context], convert_to_numpy=True)
    vecs = normalize(vecs)
    return float(np.dot(vecs[0], vecs[1]))


print(f"{'Question':<45} {'LexOverlap':>10}  {'EmbedSim':>8}  {'TopScore':>8}")
print("-" * 78)
for r in results:
    context = " ".join(r["retrieved_chunks"])
    overlap = word_overlap_ratio(r["answer"], context)
    embed_sim = embedding_grounding_score(r["answer"], context)
    flag = "  <-- check" if (overlap < 0.3 and embed_sim < 0.5) else ""
    print(f"{r['question'][:43]:<45} {overlap:>10.2f}  {embed_sim:>8.2f}  {r['scores'][0]:>8.3f}{flag}")


## 12. Conclusion

This notebook implements a Retrieval-Augmented Generation system that answers questions grounded
in a custom PDF document. The pipeline uses LangChain for document loading and chunking,
Sentence-Transformers (`all-MiniLM-L6-v2`) for embeddings, a cosine-similarity FAISS index for
retrieval, and FLAN-T5-base (via `text2text-generation`) for answer generation.

**Design decisions and why they matter:**
- Switched retrieval from raw L2 distance to cosine similarity (normalized embeddings +
  `IndexFlatIP`), which is the more standard and more reliable metric for sentence embeddings.
- Used `text2text-generation`, not `text-generation`, since FLAN-T5 is encoder-decoder; the wrong
  task silently produces empty/garbage answers rather than erroring.
- Switched generation from greedy decoding to beam search (`num_beams=4`,
  `no_repeat_ngram_size=3`) to reduce truncated or repetitive answers on longer contexts.
- Surfaced retrieval scores and chunks alongside every answer, so results are auditable.
- Added an out-of-scope control question, so the system's most important failure mode
  (hallucinating an answer that isn't in the document) is testable, not just its happy path.
- Upgraded the grounding check from pure lexical overlap to a combination of lexical overlap
  *and* embedding-based cosine similarity between answer and context, so paraphrased-but-correct
  answers aren't penalized and coincidental-word-reuse hallucinations aren't missed.

**Limitations / next steps:**
- Chunk size and overlap (500/50) were reasoned about but not empirically tuned via a
  chunking-strategy sweep.
- FLAN-T5-base is a small model; a larger instruction-tuned model would likely produce richer
  answers, at the cost of more compute.
- The embedding-based grounding check is still a heuristic (semantic similarity is not the same
  as logical entailment) -- an LLM-judge or NLI-based faithfulness check would be a stronger
  next step.
- No held-out question set with known correct answers was used for a precision/recall style
  evaluation of retrieval quality.
- This notebook was corrected and extended in a sandboxed environment without access to
  Hugging Face model downloads, so cell outputs are not populated here. Run all cells top-to-bottom
  in Colab (or any environment with internet + `pip install` access) to generate live outputs
  before submitting.